In [1]:
# === Imports ===
import torch
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.datasets as datasets
import torchvision.models as models
import torchvision.transforms as transforms

# === Dataset ===
dataset = datasets.ImageFolder(
    'dataset',
    transforms.Compose([
        transforms.ColorJitter(0.1, 0.1, 0.1, 0.1),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
)

print("Classes:", dataset.classes)  # should print ['Left', 'No_Sign', 'Right']

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [len(dataset) - 50, 50])

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=8, shuffle=True)

# === Model ===
model = models.resnet18(pretrained=True)
model.fc = torch.nn.Linear(512, 3)  # 3 classes exactly
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# === Optimizer ===
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# === Train ===
NUM_EPOCHS = 50
BEST_MODEL_PATH = 'best_model_resnet18.pth'
best_accuracy = 0.0

for epoch in range(NUM_EPOCHS):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = F.cross_entropy(outputs, labels)
        loss.backward()
        optimizer.step()

    # === Validation ===
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total
    print(f"{epoch}: {acc:.4f}")

    if acc > best_accuracy:
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        best_accuracy = acc


Classes: ['Left', 'No_Sign', 'Right']
0: 0.7800
1: 0.8800
2: 0.9600
3: 0.9800
4: 0.9800
5: 0.9800
6: 0.9800
7: 0.9800
8: 0.9800
9: 0.9800
10: 0.9800
11: 0.9800
12: 0.9800
13: 0.9800
14: 0.9800
15: 0.9800
16: 0.9800
17: 0.9800
18: 0.9800
19: 0.9800
20: 0.9600
21: 0.9800
22: 0.9800
23: 0.9800
24: 0.9600
25: 0.9800
26: 0.9800
27: 0.9800
28: 0.9800
29: 0.9800
30: 0.9800
31: 0.9800
32: 0.9800
33: 0.9800
34: 0.9800
35: 0.9800
36: 0.9800
37: 0.9800
38: 0.9800
39: 0.9800
40: 0.9800
41: 0.9800
42: 0.9800
43: 0.9800
44: 0.9800
45: 0.9800
46: 0.9600
47: 0.9800
48: 0.9800
49: 0.9800
